# Validation experiment: stability of top-contributing tokens and token removal

We reproduce stability checks for four distances used in the paper:
**Burrows’s Delta (L1), Cosine Delta, Jensen–Shannon Delta (JSD), and Rank-Turbulence Delta (RTD)**.


In [1]:
# Imports
import os
import json
import numpy as np
import pandas as pd

from scipy.special import rel_entr
from scipy.stats import rankdata


In [2]:
# ----------------------------
# Load the RU frequency matrix (as produced by `russian_tf.ipynb`)
# ----------------------------
# Expected layout (as in the project):
# df rows: tokens
# df columns: ['Count', 'Word', <doc1>, <doc2>, ...]
# Values in doc-columns are relative frequencies.
DATA_DIR = "data"
if not os.path.isdir(DATA_DIR) and os.path.isdir(os.path.join("appendix", "data")):
    os.chdir("appendix")
    DATA_DIR = "data"

df = pd.read_json(os.path.join(DATA_DIR, "TfMatrixRU.json"))
booknames = df.columns[2:]
tokens_all = df["Word"].to_numpy(dtype=object)

authors = pd.read_json(os.path.join(DATA_DIR, "authorsRU.json"), orient="index").values.reshape(-1)
authors = np.asarray(authors, dtype=object)

X_rf_all = df[booknames].to_numpy(dtype=float)  # shape: (V, D), relative frequencies

# Token-wise standard deviations are needed for (un)centred z-scores.
token_std_all = X_rf_all.std(axis=1, ddof=0)
token_std_all[token_std_all == 0] = 1.0

print(f"Loaded RU matrix: V={X_rf_all.shape[0]} tokens, D={X_rf_all.shape[1]} documents, authors={len(np.unique(authors))}.")


Loaded RU matrix: V=20000 tokens, D=639 documents, authors=89.


In [3]:
# ----------------------------
# Vector transforms (as in the paper)
# ----------------------------
def z_centered(X_rf, token_std):
    """Centred z-scores: (rf - mean)/std, token-wise."""
    mu = X_rf.mean(axis=1, keepdims=True)
    return (X_rf - mu) / token_std[:, None]

def z_uncentered(X_rf, token_std):
    """Uncentred z-scores: rf/std, token-wise."""
    return X_rf / token_std[:, None]

# ----------------------------
# Distances and per-token contributions
# ----------------------------
def dist_l1(u, v):
    """Mean L1 distance (Burrows’s Delta up to a constant factor)."""
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    return float(np.mean(np.abs(u - v)))

def dist_cosine(u, v):
    """Cosine distance: 1 - cosine similarity."""
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    nu = np.linalg.norm(u)
    nv = np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return 0.0
    return float(1.0 - np.dot(u, v) / (nu * nv))

def burrows_contrib(z1, z2):
    """Token-wise contribution for Burrows’s Delta (absolute z-score difference)."""
    return np.abs(np.asarray(z1) - np.asarray(z2))

def cosine_contrib(z1, z2):
    """Token-wise contribution proxy for Cosine Delta (as in the paper figures).

    We keep only dimensions where the (unit-normalised) components have opposite signs,
    i.e., where they *increase* cosine distance. The returned values are signed so that
    positive contributions indicate higher values in text/author 1 (analogous to other shifts).
    """
    v1 = np.asarray(z1, dtype=float).copy()
    v2 = np.asarray(z2, dtype=float).copy()
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 > 0:
        v1 = v1 / n1
    if n2 > 0:
        v2 = v2 / n2

    # Positive distance contribution occurs when v1_i * v2_i < 0 (opposite signs).
    mask = (-v1 * v2) > 0
    c = np.zeros_like(v1)
    if np.any(mask):
        c[mask] = (-v1[mask] * v2[mask]) * np.sign(v1[mask] - v2[mask])
        c[c == 0] = 0
    return c

# --- Ready-to-use functions required by the task (copied from the project notebooks) ---
def word_jsd(matrix, epsilon=1e-10):
    """Per-token (element-wise) Jensen–Shannon contribution.

    matrix: (N, 2) — columns are mean frequencies for author1 and author2.
    Returns: (N,) — signed contribution per word (positive = more in author1).
    """
    p = np.asarray(matrix[:, 0], dtype=np.float64)
    q = np.asarray(matrix[:, 1], dtype=np.float64)
    if p.sum() > 0:
        p = p / p.sum()
    else:
        p = np.ones_like(p) / len(p)
    if q.sum() > 0:
        q = q / q.sum()
    else:
        q = np.ones_like(q) / len(q)
    p = p + epsilon
    q = q + epsilon
    p, q = p / p.sum(), q / q.sum()
    m = (p + q) / 2.0
    left = rel_entr(p, m)
    right = rel_entr(q, m)
    contrib = 0.5 * left + 0.5 * right
    sign = np.sign(p - q)
    sign[sign == 0] = 1
    return sign * contrib

def word_rtd(matrix, alpha=1.0):
    """Per-token (element-wise) Rank-turbulence contribution.

    Normalised so that sum(|c_i|) equals the total rank-turbulence divergence D^R_alpha in [0, 1].

    matrix: (N, 2) — columns are mean frequencies for author1 and author2.
    Returns: (N,) — signed contribution per word (positive = more in author1).
    """
    f1 = np.asarray(matrix[:, 0])
    f2 = np.asarray(matrix[:, 1])
    V = len(f1)
    present1 = f1 > 0
    present2 = f2 > 0
    union = present1 | present2
    N1, N2 = int(present1.sum()), int(present2.sum())
    if N1 == 0 and N2 == 0:
        return np.zeros(V)

    r_missing_in_1 = N1 + N2 / 2.0
    r_missing_in_2 = N2 + N1 / 2.0

    r1 = np.full(V, np.nan)
    r2 = np.full(V, np.nan)

    if N1 > 0:
        r1[present1] = rankdata(-f1[present1], method="average")
    r1[~present1 & present2] = r_missing_in_1

    if N2 > 0:
        r2[present2] = rankdata(-f2[present2], method="average")
    r2[~present2 & present1] = r_missing_in_2

    p_exp = 1.0 / (alpha + 1.0)
    inv_r1 = np.where(union, 1.0 / r1, 0.0)
    inv_r2 = np.where(union, 1.0 / r2, 0.0)
    terms = np.where(union, np.abs(inv_r1**alpha - inv_r2**alpha) ** p_exp, 0.0)

    inv_m1 = 1.0 / r_missing_in_1 if r_missing_in_1 > 0 else 0.0
    inv_m2 = 1.0 / r_missing_in_2 if r_missing_in_2 > 0 else 0.0
    norm = 0.0
    if N1 > 0:
        norm += np.sum(np.abs(inv_r1[present1]**alpha - inv_m2**alpha) ** p_exp)
    if N2 > 0:
        norm += np.sum(np.abs(inv_m1**alpha - inv_r2[present2]**alpha) ** p_exp)

    if norm > 0:
        terms = terms / norm

    sign = np.sign(f1 - f2)
    sign[sign == 0] = 1
    return sign * np.nan_to_num(terms, nan=0.0)

def jsd_total_from_contrib(c):
    """Total JSD for a signed contribution vector produced by `word_jsd`."""
    return float(np.sum(np.abs(c)))

def rtd_total_from_contrib(c):
    """Total RTD for a signed contribution vector produced by `word_rtd`."""
    return float(np.sum(np.abs(c)))

def top_k_tokens(tokens, contrib, k=30):
    """Return top-k tokens by |contribution|."""
    idx = np.argsort(np.abs(contrib))[::-1][:k]
    return list(tokens[idx]), idx

def jaccard(a, b):
    a, b = set(a), set(b)
    if len(a | b) == 0:
        return 0.0
    return float(len(a & b) / len(a | b))


In [5]:
# ----------------------------
# Choose an author pair (defaults to two most frequent authors in the corpus)
# ----------------------------
author_counts = pd.Series(authors).value_counts()
a1, a2 = 'ТолстойЛ', 'Достоевский'
print(f"Selected author pair: {a1} vs {a2} (works: {author_counts[a1]} vs {author_counts[a2]}).")

idx1 = np.where(authors == a1)[0]
idx2 = np.where(authors == a2)[0]


Selected author pair: ТолстойЛ vs Достоевский (works: 15 vs 16).


In [34]:
# ----------------------------
# Core routine: compute contributions for a given MFW size
# ----------------------------
def compute_pair_contributions(mfw=3000, alpha=1.0):
    # Restrict to top-MFW rows (the matrix is ordered by frequency)
    tokens = tokens_all[:mfw]
    X_rf = X_rf_all[:mfw, :]
    token_std = token_std_all[:mfw]

    Zc = z_centered(X_rf, token_std)
    Zu = z_uncentered(X_rf, token_std)

    # Author mean vectors (document-average) in the corresponding representation
    zc1 = Zc[:, idx1].mean(axis=1)
    zc2 = Zc[:, idx2].mean(axis=1)

    zu1 = Zu[:, idx1].mean(axis=1)
    zu2 = Zu[:, idx2].mean(axis=1)

    out = {}

    # Burrows’s Delta (L1 on centred z-scores)
    c_l1 = burrows_contrib(zc1, zc2)
    out["burrows"] = {"tokens": tokens, "contrib": c_l1, "distance": dist_l1(zc1, zc2)}

    # Cosine Delta (cosine distance on centred z-scores)
    c_cos = cosine_contrib(zc1, zc2)
    out["cosine"] = {"tokens": tokens, "contrib": c_cos, "distance": dist_cosine(zc1, zc2)}

    # JSD
    mat = np.column_stack([zu1, zu2])

    c_jsd = word_jsd(mat)
    out["jsd"] = {"tokens": tokens, "contrib": c_jsd, "distance": jsd_total_from_contrib(c_jsd)}

    # RTD
    mat = np.column_stack([zc1, zc2])

    c_rtd = word_rtd(mat, alpha=alpha)
    out["rtd"] = {"tokens": tokens, "contrib": c_rtd, "distance": rtd_total_from_contrib(c_rtd)}

    return out

# Quick sanity check
res = compute_pair_contributions(mfw=1000, alpha=1.0)
for k in ["burrows", "cosine", "jsd", "rtd"]:
    print(k, "distance =", res[k]["distance"])


burrows distance = 0.5286395070547044
cosine distance = 0.8148638768895917
jsd distance = 0.03597162528481805
rtd distance = 0.7589657702674963


### Experiment 1: stability under small changes in MFW size

In [35]:
MFW_GRID = [2800, 3000, 3200]          # small perturbations around a typical value
TOP_K = 30

base_mfw = 3000
base = compute_pair_contributions(mfw=base_mfw, alpha=1/12)

rows = []
for mfw in MFW_GRID:
    r = compute_pair_contributions(mfw=mfw, alpha=1/12)
    for metric in ["burrows", "cosine", "jsd", "rtd"]:
        t_base, _ = top_k_tokens(base[metric]["tokens"], base[metric]["contrib"], k=TOP_K)
        t_alt, _ = top_k_tokens(r[metric]["tokens"], r[metric]["contrib"], k=TOP_K)
        rows.append({
            "metric": metric,
            "mfw": mfw,
            "jaccard(top_tokens)": jaccard(t_base, t_alt),
        })

stab_mfw = pd.DataFrame(rows).sort_values(["metric", "mfw"])
stab_mfw


,metric,mfw,jaccard(top_tokens)
0,burrows,2800,0.935484
4,burrows,3000,1.000000
8,burrows,3200,0.875000
1,cosine,2800,0.935484
5,cosine,3000,1.000000
9,cosine,3200,0.935484
2,jsd,2800,0.818182
6,jsd,3000,1.000000
10,jsd,3200,0.764706
3,rtd,2800,0.875000


### Experiment 2: stability under resampling (bootstrap over documents)

In [42]:
RNG = np.random.default_rng(0)
B = 100
TOP_K = 30
mfw = 3000
alpha = 1/12

# Full-sample top lists
full = compute_pair_contributions(mfw=mfw, alpha=alpha)
full_top = {m: top_k_tokens(full[m]["tokens"], full[m]["contrib"], k=TOP_K)[0] for m in ["burrows","cosine","jsd","rtd"]}

def bootstrap_mean(Z, doc_idx, rng):
    # Sample with replacement within an author.
    draw = rng.choice(doc_idx, size=len(doc_idx), replace=True)
    return Z[:, draw].mean(axis=1)

# Precompute representations for speed
tokens = tokens_all[:mfw]
X_rf = X_rf_all[:mfw, :]
token_std = token_std_all[:mfw]
Zc = z_centered(X_rf, token_std)
Zu = z_uncentered(X_rf, token_std)

scores = {m: [] for m in ["burrows","cosine","jsd","rtd"]}

for _ in range(B):
    zc1 = bootstrap_mean(Zc, idx1, RNG)
    zc2 = bootstrap_mean(Zc, idx2, RNG)
    zu1 = bootstrap_mean(Zu, idx1, RNG)
    zu2 = bootstrap_mean(Zu, idx2, RNG)

    # Contributions per metric
    c_b = burrows_contrib(zc1, zc2)
    c_c = cosine_contrib(zc1, zc2)
    mat = np.column_stack([zu1, zu2])
    c_j = word_jsd(mat)
    c_r = word_rtd(mat, alpha=alpha)

    tops = {
        "burrows": top_k_tokens(tokens, c_b, k=TOP_K)[0],
        "cosine": top_k_tokens(tokens, c_c, k=TOP_K)[0],
        "jsd": top_k_tokens(tokens, c_j, k=TOP_K)[0],
        "rtd": top_k_tokens(tokens, c_r, k=TOP_K)[0],
    }

    for metric in scores:
        scores[metric].append(jaccard(full_top[metric], tops[metric]))

stab_boot = pd.DataFrame({
    "metric": list(scores.keys()),
    "mean_jaccard": [float(np.mean(scores[m])) for m in scores],
    "std_jaccard": [float(np.std(scores[m])) for m in scores],
}).sort_values("metric")
stab_boot


,metric,mean_jaccard,std_jaccard
0,burrows,0.411823,0.073938
1,cosine,0.495296,0.093493
2,jsd,0.459939,0.101857
3,rtd,0.370378,0.074901


### Experiment 3: remove top contributors and re-measure separation

In [32]:
mfw = 3000
alpha = 1/12
TOP_REMOVE = [10, 50, 100]

res_full = compute_pair_contributions(mfw=mfw, alpha=alpha)

# Build the underlying vectors once (we need them for recomputing distances after removal).
tokens = tokens_all[:mfw]
X_rf = X_rf_all[:mfw, :]
token_std = token_std_all[:mfw]
Zc = z_centered(X_rf, token_std)
Zu = z_uncentered(X_rf, token_std)

zc1 = Zc[:, idx1].mean(axis=1)
zc2 = Zc[:, idx2].mean(axis=1)
zu1 = Zu[:, idx1].mean(axis=1)
zu2 = Zu[:, idx2].mean(axis=1)

def remove_dims(vec, rm_idx):
    v = np.asarray(vec, dtype=float).copy()
    v[rm_idx] = 0.0
    return v

rows = []
for k in TOP_REMOVE:
    # Burrows
    top_tokens_b, rm_b = top_k_tokens(tokens, burrows_contrib(zc1, zc2), k=k)
    d0 = dist_l1(zc1, zc2)
    d1 = dist_l1(remove_dims(zc1, rm_b), remove_dims(zc2, rm_b))
    rows.append({"metric": "burrows", "removed_top_k": k, "distance_before": d0, "distance_after": d1})

    # Cosine
    top_tokens_c, rm_c = top_k_tokens(tokens, cosine_contrib(zc1, zc2), k=k)
    d0 = dist_cosine(zc1, zc2)
    d1 = dist_cosine(remove_dims(zc1, rm_c), remove_dims(zc2, rm_c))
    rows.append({"metric": "cosine", "removed_top_k": k, "distance_before": d0, "distance_after": d1})

    # JSD
    c_jsd = word_jsd(np.column_stack([zu1, zu2]))
    top_tokens_j, rm_j = top_k_tokens(tokens, c_jsd, k=k)
    d0 = jsd_total_from_contrib(c_jsd)
    d1 = jsd_total_from_contrib(word_jsd(np.column_stack([remove_dims(zu1, rm_j), remove_dims(zu2, rm_j)])))
    rows.append({"metric": "jsd", "removed_top_k": k, "distance_before": d0, "distance_after": d1})

    # RTD
    c_rtd = word_rtd(np.column_stack([zu1, zu2]), alpha=alpha)
    top_tokens_r, rm_r = top_k_tokens(tokens, c_rtd, k=k)
    d0 = rtd_total_from_contrib(c_rtd)
    d1 = rtd_total_from_contrib(word_rtd(np.column_stack([remove_dims(zu1, rm_r), remove_dims(zu2, rm_r)]), alpha=alpha))
    rows.append({"metric": "rtd", "removed_top_k": k, "distance_before": d0, "distance_after": d1})

removal = pd.DataFrame(rows)
removal["relative_change"] = (removal["distance_after"] - removal["distance_before"]) / removal["distance_before"]
removal.sort_values(["metric", "removed_top_k"])


,metric,removed_top_k,distance_before,distance_after,relative_change
0,burrows,10,0.429077,0.420302,-0.020450
4,burrows,50,0.429077,0.393577,-0.082736
8,burrows,100,0.429077,0.367318,-0.143934
1,cosine,10,0.815946,0.793968,-0.026936
5,cosine,50,0.815946,0.736434,-0.097448
9,cosine,100,0.815946,0.687133,-0.157870
2,jsd,10,0.056933,0.054729,-0.038707
6,jsd,50,0.056933,0.049241,-0.135103
10,jsd,100,0.056933,0.044808,-0.212964
3,rtd,10,0.222510,0.220171,-0.010512


## Notes on interpretation

- The *MFW stability* table reports the Jaccard overlap of top contributing tokens (by absolute contribution) between the baseline MFW setting and small perturbations.
- The *bootstrap stability* table reports the mean and standard deviation of the same overlap under bootstrap resampling of documents within each author.
- The *token removal* table demonstrates the expected directional effect: removing the top contributors (identified at the baseline setting) decreases the corresponding inter-author separation for all four distances.
